# 01 — Inspection Silver Delta Lake

**Objectif** : inspecter le schéma, les métadonnées et un échantillon de la table `silver.trays`  
**Source** : ADLS Gen2 — `dlsecatcandlingfrcedev` / container `silver` / table `trays`  
**Dernière mise à jour** : 2026-06-22

In [1]:
import os
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

load_dotenv()
print("OK")

OK


In [2]:
STORAGE_ACCOUNT = os.getenv("SILVER_STORAGE_ACCOUNT_NAME")
STORAGE_KEY      = os.getenv("SILVER_STORAGE_ACCOUNT_KEY")
CONTAINER        = os.getenv("SILVER_CONTAINER", "silver")
TABLE_PATH       = os.getenv("SILVER_TABLE_PATH", "trays")

TABLE_URI = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{TABLE_PATH}"

STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT,
    "account_key":  STORAGE_KEY,
}

dt = DeltaTable(TABLE_URI, storage_options=STORAGE_OPTIONS)
print(f"Connecté — version Delta : {dt.version()}")
print(f"URI : {TABLE_URI}")

Connecté — version Delta : 13
URI : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays


In [3]:
schema = dt.schema()
schema_df = pd.DataFrame([
    {"name": f.name, "type": str(f.type), "nullable": f.nullable}
    for f in schema.fields
])
schema_df

,name,type,nullable
0,tray_id,"PrimitiveType(""string"")",False
1,machine_id,"PrimitiveType(""string"")",False
2,candled_at,"PrimitiveType(""timestamp"")",False
3,candled_date,"PrimitiveType(""date"")",False
4,flock,"PrimitiveType(""string"")",False
5,trolley,"PrimitiveType(""string"")",False
6,tray_seq,"PrimitiveType(""integer"")",False
7,flock_name,"PrimitiveType(""string"")",True
8,trolley_name,"PrimitiveType(""string"")",True
9,caliber,"PrimitiveType(""string"")",True


In [4]:
meta = dt.metadata()
print(f"Partition columns : {meta.partition_columns}")
print(f"Created at        : {meta.created_time}")
print(f"Delta version     : {dt.version()}")

Partition columns : ['machine_id', 'candled_date']
Created at        : 1780480109868
Delta version     : 13


In [5]:
MACHINE_ID   = "PMAF-C012501"
CANDLED_DATE = "2026-05-15"

df = dt.to_pandas(
    partitions=[
        ("machine_id",   "=", MACHINE_ID),
        ("candled_date", "=", CANDLED_DATE),
    ]
)

print(f"Lignes chargées : {len(df)}")
print(f"Colonnes        : {list(df.columns)}")
df.head(3)

Lignes chargées : 1344
Colonnes        : ['tray_id', 'machine_id', 'candled_at', 'candled_date', 'flock', 'trolley', 'tray_seq', 'flock_name', 'trolley_name', 'caliber', 'setter_tray_number_flock', 'n_total', 'n_fertile', 'n_clear', 'n_early_dead', 'n_late_dead', 'n_missing', 'is_count_consistent', 'matrix_compact', 'light_flat', 'alarm_emergency_stop', 'alarm_air_pressure_fault', 'processed_at', 'bronze_path']


,tray_id,machine_id,candled_at,candled_date,flock,trolley,tray_seq,flock_name,trolley_name,caliber,...,n_early_dead,n_late_dead,n_missing,is_count_consistent,matrix_compact,light_flat,alarm_emergency_stop,alarm_air_pressure_fault,processed_at,bronze_path
0,e5421d46d8e3e00b368f284d2da88b869f2be9d709a8ac...,PMAF-C012501,2026-05-15 04:33:13.105761+00:00,2026-05-15,1,20713024,5,43e6ab99aeea0b2d6ca,20713024,1,...,0,0,0,True,1111111111331111111111113313111113131311111131...,"[160145, 197634, 160347, 160778, 185549, 12317...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...
1,b1309544b86d9b1255ed3a30eacb5aefddd0feaab5ad6e...,PMAF-C012501,2026-05-15 04:34:18.744640+00:00,2026-05-15,1,20713024,16,43e6ab99aeea0b2d6ca,20713024,1,...,1,1,0,True,1111111111111123111111111131111311111111111111...,"[159991, 185577, 185489, 37500, 186016, 184573...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...
2,bb953663c1a9a7a7e9f67f142fbcc81170650b6aca3ca1...,PMAF-C012501,2026-05-15 04:34:31.099685+00:00,2026-05-15,1,20713024,18,43e6ab99aeea0b2d6ca,20713024,1,...,3,7,0,True,1111111311133414131111311213111111111111111111...,"[184147, 160629, 160723, 160729, 147549, 18421...",0,0,2026-06-03 09:57:02.303577+00:00,raw/trolley/year=2026/month=05/day=15/PMAF-C01...


In [6]:
schema_export = {
    "exported_at":       datetime.utcnow().isoformat() + "Z",
    "table_uri":         TABLE_URI,
    "table_name":        f"{CONTAINER}.{TABLE_PATH}",
    "partition_columns": meta.partition_columns,
    "delta_version":     dt.version(),
    "fields": [
        {"name": f.name, "type": str(f.type), "nullable": f.nullable}
        for f in schema.fields
    ]
}

output_path = Path("../schemas") / f"silver_{TABLE_PATH}_schema.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schema_export, f, indent=2, ensure_ascii=False)

print(f"Schema exporté : {output_path}")

Schema exporté : ..\schemas\silver_trays_schema.json


C:\Users\u279306\AppData\Local\Temp\ipykernel_26804\3331963323.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "exported_at":       datetime.utcnow().isoformat() + "Z",
